# Training on images and spectra together

This notebook trains a small PyTorch model on a multimodal HATS catalog, where every row holds both
an image and a spectrum of the same galaxy. The catalog is a crossmatch of Legacy Survey DR10 images
with DESI EDR spectra, hosted on Hugging Face.

The model is not meant to do real science. It shows how the different column types of a catalog
reach a model:

- **Images** are fixed-shape tensor columns (`tensor[float, (160, 160)]`), one per band.
- **Spectra** are a nested column, with one row of `flux`, `lambda`, `ivar`, ... per object.

Data is streamed with `CatalogDataset`, and several PyTorch workers load batches in parallel while
the model trains.

This notebook is pre-executed because it reads data over the network.

In [1]:
import time

import lsdb
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

from lsdb_ml import CatalogDataLoader, CatalogDataset

## Load the catalog

We open only the columns we need, the `g` and `r` band images and the spectra, and keep the first
32 of its 548 partitions so the notebook runs in a few minutes.

In [2]:
catalog = lsdb.open_catalog(
    "hf://datasets/brantd/multimodal_hats_example",
    columns=["image_flux_g", "image_flux_r", "spectrum"],
)
catalog = catalog.partitions[:32]
catalog

,image_flux_g,image_flux_r,ra_mmu_legacysurvey_dr10_south_21,dec_mmu_legacysurvey_dr10_south_21,spectrum
npartitions=32,,,,,
"Order: 7, Pixel: 41472","tensor[float, (160, 160)]","tensor[float, (160, 160)]",double[pyarrow],double[pyarrow],"nested<flux: [float], ivar: [float], lsf_sigma..."
...,...,...,...,...,...
"Order: 7, Pixel: 41602",...,...,...,...,...
"Order: 7, Pixel: 41603",...,...,...,...,...


## Turn rows into tensors

The default `collate_catalog` handles flat and nested numeric columns, but not tensor columns, so
we write a `collate_fn`. It receives a `NestedFrame` with one batch of rows and returns tensors:

- `to_stack()` turns a tensor column into one `(batch, 160, 160)` array. We stack the two bands
  into channels.
- Every DESI spectrum is sampled on the same wavelength grid, so the flat `flux` values reshape into
  a `(batch, n_wavelengths)` array.

In [3]:
def collate(frame):
    image = np.stack([frame["image_flux_g"].to_stack(), frame["image_flux_r"].to_stack()], axis=1)
    spectrum = frame["spectrum"].nest["flux"].to_numpy(dtype=np.float32).reshape(len(frame), -1)
    return {"image": torch.from_numpy(image), "spectrum": torch.tensor(spectrum)}


batch = collate(catalog.partitions[0].compute())
{name: tensor.shape for name, tensor in batch.items()}

Building task graph...


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

{'image': torch.Size([7, 2, 160, 160]), 'spectrum': torch.Size([7, 7781])}

## Stream batches with several workers

With `num_workers > 0`, each PyTorch worker streams its own share of the partitions with dask's
synchronous scheduler, so no dask client is needed.

We start the workers with `forkserver`, because forking the notebook process directly (the default
on Linux) can hang. The dataset, including our `collate` function, is sent to each worker with
`cloudpickle`, so functions defined in a notebook work.

> **Faster worker startup.** Each worker imports torch and LSDB when it starts. To do this only
> once, preload them into the fork server:
> `ctx = multiprocessing.get_context("forkserver")`, `ctx.set_forkserver_preload(["torch", "lsdb"])`,
> and pass `multiprocessing_context=ctx`. **On macOS, preload only `torch`** when reading over HTTPS
> (e.g. `hf://`): with `lsdb`, `nested_pandas` or `pandas` preloaded, workers segfault on their
> first request, because the macOS certificate check (`truststore`) is not fork-safe. Here, reading
> from Hugging Face takes much longer than startup, so we skip it.

In [4]:
dataset = CatalogDataset(catalog, batch_size=64, seed=0, drop_last=True, collate_fn=collate)
loader = CatalogDataLoader(dataset, num_workers=4, multiprocessing_context="forkserver")

## A two-tower model

Each modality gets its own small encoder: a 2D CNN for the images and a 1D CNN for the spectra.
Both map to a shared 64-dimensional embedding. A `BatchNorm` at the input of each encoder takes
care of the very different flux scales of the two.

We train it contrastively, as in CLIP or AstroCLIP: in a batch of `B` objects, the image of each
object should be closer to its own spectrum than to the other `B - 1` spectra. No labels are
needed.

In [5]:
class TwoTower(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.image = nn.Sequential(
            nn.BatchNorm2d(2),
            nn.Conv2d(2, 16, 5, stride=4),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=4),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, dim),
        )
        self.spectrum = nn.Sequential(
            nn.BatchNorm1d(1),
            nn.Conv1d(1, 16, 9, stride=8),
            nn.ReLU(),
            nn.Conv1d(16, 32, 9, stride=8),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, dim),
        )

    def forward(self, image, spectrum):
        return F.normalize(self.image(image), dim=1), F.normalize(self.spectrum(spectrum[:, None]), dim=1)

## Train

`CatalogDataLoader` gives each epoch a new shuffle order. Besides the loss, we track how often an
image's nearest spectrum in the batch is its own; chance is `1 / 64`. We also log how long each
batch took, for the throughput report at the end.

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
model = TwoTower().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def train_step(batch):
    image_emb, spectrum_emb = model(batch["image"].to(device), batch["spectrum"].to(device))
    logits = image_emb @ spectrum_emb.T / 0.1
    target = torch.arange(len(logits), device=device)
    loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # .item() waits for the GPU, so timing a call to train_step includes its GPU work.
    return loss.item(), (logits.argmax(1) == target).float().mean().item()


log = []  # one row per batch
for epoch in range(3):
    last = time.perf_counter()
    for batch in loader:
        loaded = time.perf_counter()
        loss, matched = train_step(batch)
        done = time.perf_counter()
        log.append(dict(epoch=epoch, wait_s=loaded - last, step_s=done - loaded, loss=loss, matched=matched))
        last = done
    epoch_log = pd.DataFrame(log).query("epoch == @epoch")
    print(f"epoch {epoch}: loss {epoch_log.loss.mean():.3f}, matched {epoch_log.matched.mean():.1%}")

epoch 0: loss 3.883, matched 4.7%
epoch 1: loss 3.631, matched 5.5%
epoch 2: loss 3.515, matched 5.7%


## Throughput report

Is the GPU waiting on LSDB, or LSDB on the GPU? Three numbers tell us:

1. **Data rate.** Objects per second that the loader delivered during training, and the size of
   those tensors in MB per second. (This is the in-memory size, not the compressed bytes read.)
2. **GPU ceiling.** Objects per second the training step can process when the data is already in
   memory, measured by repeating the step on one batch.
3. **GPU-busy fraction.** The share of wall time the loop spent in the training step. The rest was
   spent waiting for the next batch.

If the GPU-busy fraction is close to 1, the model is the bottleneck and faster loading would not
help. If it is low, the GPU sits idle and training runs only as fast as the data arrives.

The step time includes copying the batch to the device and some Python overhead, so the busy
fraction is an upper bound on how busy the GPU itself was. Without a GPU, the model trains on the CPU, and
"GPU" in this section means the CPU: the ceiling is lower, and the training step shares the cores
with the loader workers.

In [7]:
train_step(batch)  # warm up; these extra steps also train the model a little
start = time.perf_counter()
for _ in range(20):
    train_step(batch)
gpu_ceiling = 20 * len(batch["image"]) / (time.perf_counter() - start)
print(f"GPU ceiling: {gpu_ceiling:,.0f} objects/s on {device}")

GPU ceiling: 4,356 objects/s on cuda


Each epoch's numbers come from the timings logged during training. `startup_s` is the wait for
the first batch, which includes starting the workers and reading their first partitions.
`gpu_headroom` is how many times faster the GPU could train than the data arrived.

In [8]:
mb_per_object = sum(tensor[0].nbytes for tensor in batch.values()) / 1e6
report = (
    pd.DataFrame(log)
    .groupby("epoch")
    .agg(
        batches=("step_s", "size"),
        startup_s=("wait_s", "first"),
        wait_s=("wait_s", "sum"),
        step_s=("step_s", "sum"),
    )
)
wall_s = report.wait_s + report.step_s
report["objects_per_s"] = report.batches * len(batch["image"]) / wall_s
report["MB_per_s"] = report.objects_per_s * mb_per_object
report["gpu_busy"] = report.step_s / wall_s
report["gpu_headroom"] = gpu_ceiling / report.objects_per_s
report.drop(columns=["wait_s", "step_s"]).round(2)

,batches,startup_s,objects_per_s,MB_per_s,gpu_busy,gpu_headroom
epoch,,,,,,
0,164,46.13,28.28,6.67,0.01,154.01
1,163,55.03,28.82,6.80,0.01,151.13
2,164,16.32,22.59,5.33,0.01,192.78


### Reading the results

*Results above from Gondor (L40 GPU). Below text is out of date
  
This notebook ran on an Apple laptop (MPS), reading from Hugging Face over a home internet
connection. There, the pipeline is entirely **data-bound**: the GPU was busy about 1% of the time
and could have trained a couple of hundred times faster than the data arrived. Waiting for the
first batch of each epoch, while the workers start and read their first partitions, took up to a
fifth of the epoch.

The data rate is set by how fast partitions can be downloaded, not by LSDB's CPU work: each
partition is about 100 MB of images and spectra and arrives as a whole. In a short test on the same
machine, using 1 to 8 workers did not raise the rate. Your numbers will differ, so check the
`gpu_busy` column on your own hardware. To move the bottleneck toward the GPU:

- **Read from faster storage.** A local copy of the catalog (e.g. on a cluster's file system)
  removes the network limit. This is by far the largest lever.
- **Then add workers.** Once reads are fast, decompression and `collate` become the CPU cost, and
  that work spreads across `num_workers`.
- **Larger models change the balance.** This model is tiny. A real image encoder does far more work
  per object, which raises `gpu_busy` without any change to the data side.